In [2]:
import os
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path: ").strip().strip('"')
    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part4a_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
districts = pd.read_csv(RAW_DIR / "districts.csv")
age_sex = pd.read_csv(RAW_DIR / "age_sex.csv")

# 1. Build 1 row per district (all 24 districts, hypothetical dataset)
pop25_tot = age_sex.groupby("district_id")["population_2025"].sum()
u15_tot = age_sex[age_sex["age_start"] < 15].groupby("district_id")["population_2025"].sum()
o65_tot = age_sex[age_sex["age_start"] >= 65].groupby("district_id")["population_2025"].sum()

df_4a = districts[["district_id", "district_name", "region", "population_2026",
                   "arrivals_2025", "departures_2025", "piped_water_pct"]].copy()
df_4a["pop_2025"] = df_4a["district_id"].map(pop25_tot)
df_4a["share_u15_pct"] = (df_4a["district_id"].map(u15_tot) / df_4a["pop_2025"]) * 100.0
df_4a["share_65p_pct"] = (df_4a["district_id"].map(o65_tot) / df_4a["pop_2025"]) * 100.0
exposure_2025 = 0.5 * (df_4a["pop_2025"] + df_4a["population_2026"])
df_4a["nmr_2025_per_1000"] = ((df_4a["arrivals_2025"] - df_4a["departures_2025"]) / exposure_2025) * 1000.0

# 2. Median imputation for the 2 missing piped_water_pct values (D04, D17)
median_water = df_4a["piped_water_pct"].median()
df_4a["piped_water_imputed"] = df_4a["piped_water_pct"].fillna(median_water)

feat_cols = ["share_u15_pct", "share_65p_pct", "nmr_2025_per_1000", "piped_water_imputed"]

# 3. Standardise features (district_id excluded because it is a nominal identifier)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_4a[feat_cols])
z_cols = [f"z_{c}" for c in feat_cols]
df_4a[z_cols] = X_scaled

# 4. Outlier diagnostic summary (IQR fences and max |z|)
outlier_rows = []
for c, zc in zip(feat_cols, z_cols):
    q1, q3 = df_4a[c].quantile(0.25), df_4a[c].quantile(0.75)
    iqr = q3 - q1
    low_f, high_f = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df_4a[c] < low_f) | (df_4a[c] > high_f)).sum())
    outlier_rows.append({
        "feature": c,
        "mean": df_4a[c].mean(),
        "std": df_4a[c].std(ddof=1),
        "median": df_4a[c].median(),
        "iqr_outliers": n_out,
        "max_abs_z": df_4a[zc].abs().max(),
    })

outlier_df = pd.DataFrame(outlier_rows)

# Show tables first, then save
print("\n=== OUTLIER & DISTRIBUTION SUMMARY (N=24) ===")
display(outlier_df)
print("=== PREPARED DISTRICT FEATURES (FIRST 10 ROWS) ===")
display(df_4a[["district_id", "region"] + feat_cols].head(10))

safe_save(lambda p: df_4a.to_csv(p, index=False), "part4a_district_features_24.csv")
safe_save(lambda p: outlier_df.to_csv(p, index=False), "part4a_outlier_summary.csv")

BASE_DIR: C:\Users\Lennon\Documents\Dummy Run

=== OUTLIER & DISTRIBUTION SUMMARY (N=24) ===


,feature,mean,std,median,iqr_outliers,max_abs_z
0,share_u15_pct,17.121943,8.685451,12.877711,0,1.509412
1,share_65p_pct,18.348345,6.955920,14.576367,0,1.509808
2,nmr_2025_per_1000,2.521942,6.861099,-1.187919,0,1.707504
3,piped_water_imputed,56.170833,17.409330,54.000000,0,1.779591


=== PREPARED DISTRICT FEATURES (FIRST 10 ROWS) ===


,district_id,region,share_u15_pct,share_65p_pct,nmr_2025_per_1000,piped_water_imputed
0,D01,Region 1,29.274841,14.633651,-2.692109,31.3
1,D02,Region 1,13.292814,12.408766,9.584862,80.5
2,D03,Region 1,9.282863,27.971813,-2.322536,56.6
3,D04,Region 2,27.882786,14.387271,-3.666953,54.0
4,D05,Region 2,12.966073,12.417675,13.990632,77.4
5,D06,Region 2,9.518690,28.199916,-1.265481,50.4
6,D07,Region 3,29.459441,14.586691,-2.285688,44.5
7,D08,Region 3,12.865825,12.428101,12.575920,72.9
8,D09,Region 3,9.026983,27.839942,-1.420767,53.6
9,D10,Region 4,29.132085,14.566042,-0.938212,33.5


Saved: C:\Users\Lennon\Documents\Dummy Run\outputs\part4a_district_features_24.csv
Saved: C:\Users\Lennon\Documents\Dummy Run\outputs\part4a_outlier_summary.csv
